# TalentDesk, Module 3 Section 3 Lab (Exercise): Test-Driven Iteration and Headless CI

A hands-on exercise on making code work with Claude reliable. It combines the two M3S3 skills: driving
an implementation **test-first** from concrete examples and asking clarifying questions on a fuzzy spec
(Lab 1), and running Claude as a **CI step** that emits **schema-valid JSON**, reviews against
CLAUDE.md, reports **incrementally**, and gates the build (Lab 2). You fill in four short `TODO` blocks;
everything else is provided. A real **pytest** loop and JSON-schema validation run offline, and live
cells use your **Anthropic API key** with **Sonnet** (`claude-sonnet-4-6`).

## The real-world scenario

TalentDesk needs a candidate-eligibility function, and "make eligibility work" invites the wrong guess.
A few concrete input/output examples and a failing test suite pin down exactly what "correct" means, and
when the spec is fuzzy the fastest path is for Claude to ask a couple of sharp questions first, not to
code blindly.

Separately, TalentDesk wants every pull request auto-reviewed. A human-in-the-terminal session will not
do; CI needs a command that runs unattended, emits **parseable JSON** a script can branch on, applies the
same CLAUDE.md rules on every machine, and on re-runs does not spam the same findings twice.

The question this lab answers: **how do examples and tests remove ambiguity, and how do you turn Claude
into a CI step whose output your pipeline can trust?**

## Objectives

- Turn **input/output examples** into a failing test suite, then make it pass (test-driven iteration).
- Use an **interview** gate so a fuzzy spec produces questions instead of a wrong guess.
- Produce **schema-valid JSON** for a CI review, keep **review** separate from **generation**, report
  **incrementally**, and gate the build on severity.

## The outcome you should reach

By the end you will have:

- a test suite generated from examples that goes red on the stub and green on your implementation;
- an interview gate that lists what to ask about a fuzzy spec;
- a review payload that validates against a JSON schema (and a bad one that is rejected);
- an incremental review that labels findings NEW vs STILL OPEN, and a build gate that fails on a
  high-severity finding.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all testable offline. The live cells need a
real key.

## How to run

Run top to bottom. The pytest loop, the schema validation, and the pipeline simulation run anywhere. The
live cells call Claude for the interview and structured output, so paste a real key into **Setup 2/3**
and re-run from the top; otherwise they skip.

## 0. Setup

**This cell:** installs the packages. `pytest` runs the test loop, `jsonschema` validates the
structured output, and the `anthropic` SDK drives the live cells.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q anthropic python-dotenv pytest jsonschema

**This cell:** imports, the model, the `RUN_LIVE` switch, and a client for the live cells.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, a client =====
import os                                       # filesystem paths and env for pytest
import sys                                       # run pytest as a subprocess with this interpreter
import subprocess                               # invoke pytest and capture its output
import re                                       # strip code fences from model output
import json                                     # parse and print structured output
import inspect                                  # read your implementation's source for the green run
import textwrap                                  # keeps the embedded file bodies readable
import jsonschema                               # validate output against the schema

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cells will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

client = None                                    # the live cells build this only if RUN_LIVE
if RUN_LIVE:
    import anthropic
    client = anthropic.Anthropic()

print("live model calls:", "ON" if RUN_LIVE else "OFF (pytest + schema cells run offline)")

**This cell:** writes a tiny **TalentDesk package** with a deliberately empty `meets_bar` (a stub),
so the tests we add next fail first. This is the starting point of the test-first loop.

In [ ]:
# ===== SETUP 3/3 - create the package with an unimplemented stub =====
REPO = os.path.join(os.getcwd(), "talentdesk_tdd")

def write(rel, content):
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

write("talentdesk/__init__.py", "")               # make talentdesk an importable package
write("talentdesk/screening.py", textwrap.dedent("""\
    def meets_bar(candidate):
        # not implemented yet (this is the stub)
        return None
    """))
print("created package at", REPO)

### Examples first, then tests

Concrete **input/output examples** are the cheapest way to remove ambiguity. Before writing code, agree
on what the function returns for specific inputs, including the tricky boundary (exactly 3 years). Those
examples become your tests, and the tests become the definition of done.

---

### 🎯 Part A - let examples and tests drive the code

**This cell:** the **input/output examples** (provided). The rule: eligible only if the candidate
has at least 3 years (inclusive) **and** a complete application.

In [ ]:
# ===== the input/output examples that define "correct" (provided) =====
EXAMPLES = [
    ({"years": 5, "complete": True},  True),     # inside the bar
    ({"years": 3, "complete": True},  True),     # exactly 3 years -> inclusive
    ({"years": 2, "complete": True},  False),    # below the bar
    ({"years": 5, "complete": False}, False),    # application not complete
]
for candidate, expected in EXAMPLES:
    print(f"  {candidate} -> {expected}")

**TODO 1 (about 4 minutes).** Complete `generate_tests()`, which turns the examples into a pytest
file: one assertion per example. Writing the tests before the code is the heart of test-driven
iteration.

In [ ]:
# ===== TODO 1 - generate the test file from the examples =====
def generate_tests(examples):                      # examples -> pytest source text
    lines = ["from talentdesk.screening import meets_bar", "", "def test_examples():"]
    for candidate, expected in examples:
        # 👉 TODO 1: append one assertion per example, e.g.
        #    lines.append(f"    assert meets_bar({candidate!r}) is {expected}")
        pass
    return "\n".join(lines) + "\n"

test_src = generate_tests(EXAMPLES)
write("tests/test_screening.py", test_src)
print(test_src)

**Self-check (offline).** One assertion per example, importing the function under test.

In [ ]:
# ===== self-check for TODO 1 =====
assert test_src.count("assert meets_bar(") == len(EXAMPLES), "one assertion per example"
assert "from talentdesk.screening import meets_bar" in test_src, "the test imports the function"
assert " is True" in test_src and " is False" in test_src, "assert exact booleans"
print("TODO 1 checks passed")

**This cell:** the pytest runner and the **RED** step (provided). Against the stub (which returns
`None`), every example fails. The tests describe the goal and prove we are not there yet.

In [ ]:
# ===== RED: run the generated tests against the stub (provided) =====
def run_pytest():
    env = {**os.environ, "PYTHONPATH": REPO}
    r = subprocess.run([sys.executable, "-m", "pytest", "-q"],
                       cwd=REPO, capture_output=True, text=True, env=env)
    return r.stdout[-600:]

print(run_pytest())

**TODO 2 (about 4 minutes).** Implement `meets_bar()` from the examples: return `True` only if the
candidate has at least 3 years **and** a complete application. Fill in the body inside the `IMPL` string;
it is both run here (for the self-check) and written to the package for the green pytest run.

In [ ]:
# ===== TODO 2 - implement meets_bar from the examples =====
IMPL = """def meets_bar(candidate):
    # 👉 TODO 2: return True only when years >= 3 AND the application is complete
    #    return candidate["years"] >= 3 and candidate["complete"] is True
    return None
"""
exec(IMPL)   # defines meets_bar in this notebook so the self-check can call it

**Self-check (offline).** Your function should match every example exactly.

In [ ]:
# ===== self-check for TODO 2 =====
for candidate, expected in EXAMPLES:
    got = meets_bar(candidate)
    assert got is expected, f"{candidate}: expected {expected}, got {got}"
print("TODO 2 checks passed")

**This cell:** the **GREEN** step (provided). It writes your implementation into the package and
reruns pytest. With the rule correct, every example passes and the red-to-green loop is closed.

In [ ]:
# ===== GREEN: write your implementation, then re-run pytest (provided) =====
write("talentdesk/screening.py", IMPL)
print(run_pytest())

**This cell:** the **interview** gate (provided). When a request is fuzzy, coding immediately bakes
in a guess. This checks a spec for the aspects that matter and lists the questions to ask first.

In [ ]:
# ===== interview gate: what to ask before coding a fuzzy spec (provided) =====
ASPECTS = {
    "boundary": "Is the years requirement inclusive or exclusive at the threshold?",
    "complete": "What counts as a complete application?",
    "status":   "Do withdrawn or on-hold candidates count?",
    "override": "Can a recruiter override the bar for a strong candidate?",
}
def interview(spec):                               # spec text -> the clarifying questions it leaves open
    return [q for key, q in ASPECTS.items() if key not in spec.lower()]

for q in interview("Make eligibility work for candidates with enough experience."):
    print("  ?", q)

**This cell:** combined vs sequential fixes (provided). Interdependent fixes belong in one pass so
they stay consistent; independent fixes are safer one at a time.

In [ ]:
# ===== combined (interdependent) vs sequential (independent) (provided) =====
def strategy(issues):                              # issues: list of (name, depends_on_another)
    return "combined" if any(dep for _, dep in issues) else "sequential"

linked = [("rename MIN_YEARS", True), ("update callers of the constant", True)]
independent = [("fix a docstring typo", False), ("add a missing type hint", False)]
print("  linked      ->", strategy(linked))
print("  independent ->", strategy(independent))

**This cell:** the live **interview-style** prompt (provided). Told to ask before coding, Claude
should return questions on the fuzzy spec rather than a confident wrong answer. Offline it prints the
expected behaviour.

In [ ]:
# ===== live: ask clarifying questions before coding =====
INTERVIEW_SYS = ("You are a careful engineer. If a request is ambiguous, ask up to 3 clarifying "
                 "questions BEFORE writing any code. Do not write code yet.")
if RUN_LIVE:
    resp = client.messages.create(model=MODEL, max_tokens=400, system=INTERVIEW_SYS,
        messages=[{"role": "user", "content": "Make eligibility work for experienced candidates."}])
    print("".join(b.text for b in resp.content if b.type == "text").strip()[:600])
else:
    print("[offline] expected: Claude asks about the years boundary, what 'complete' means, and overrides.")

---

### 🎯 Part B - make the CI output a contract

`claude -p` runs non-interactively; `--output-format json` plus `--json-schema` constrain the answer to
your shape, landing the conforming data in a `structured_output` field your script reads by key. Skipping
`--bare` keeps CLAUDE.md rules enforced in CI.

**This cell:** the **review schema** (provided): a `findings` array, each with a file, line,
severity enum, and message, forbidding extra keys. This is the contract the CLI `--json-schema` flag and
the API structured-output feature enforce.

In [ ]:
# ===== the review schema (the contract, provided) =====
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "findings": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "file":     {"type": "string"},
                "line":     {"type": "integer"},
                "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                "message":  {"type": "string"},
            },
            "required": ["file", "line", "severity", "message"],
            "additionalProperties": False,
        }},
    },
    "required": ["findings"],
    "additionalProperties": False,
}
print("schema requires:", REVIEW_SCHEMA["required"])

**This cell:** a **pipeline simulation** and validation (provided). It stands in for the headless
CLI, returning the same envelope (`result` plus `structured_output`), validates the payload, and shows a
malformed one (bad severity) being rejected.

In [ ]:
# ===== simulate the headless run, validate, and reject a bad payload (provided) =====
def simulate_pipeline_step():                      # stand-in for: claude -p ... --output-format json --json-schema
    structured = {"findings": [
        {"file": "talentdesk/screening.py", "line": 1, "severity": "high",
         "message": "Hardcoded API key; move it to an environment variable."},
        {"file": "talentdesk/screening.py", "line": 4, "severity": "medium",
         "message": "screen_candidate does not check the bar (MIN_YEARS)."},
    ]}
    return {"result": "review complete", "structured_output": structured}

envelope = simulate_pipeline_step()
jsonschema.validate(envelope["structured_output"], REVIEW_SCHEMA)      # raises if the shape is wrong
print("payload valid; findings:", len(envelope["structured_output"]["findings"]))

bad = {"findings": [{"file": "x.py", "line": 1, "severity": "critical", "message": "?"}]}  # not in enum
try:
    jsonschema.validate(bad, REVIEW_SCHEMA); print("unexpectedly valid")
except jsonschema.ValidationError as e:
    print("bad payload rejected as expected:", e.message[:60])

**This cell:** why **review** and **generation** stay separate (provided). A reviewer that inherits
the author's context also inherits its justifications and blind spots; a fresh, rule-driven session gives
an independent check.

In [ ]:
# ===== session isolation: review is not primed by generation (provided) =====
generation_ctx = ["wrote screen_candidate", "decided to skip the bar check for speed"]
review_ctx = ["CLAUDE.md rules only"]
print("context shared between the two sessions:", set(generation_ctx) & set(review_ctx) or "none")
print("the reviewer judges the code against the rules, not the author's reasoning")

**TODO 3 (about 5 minutes).** Complete `incremental_review()`. Using a stable `key()` for each
finding, label every current finding `"NEW"` or `"STILL OPEN"` (was it in the previous run?), and count
the **resolved** ones (in previous, not in current). This keeps a re-run from repeating noise.

In [ ]:
# ===== TODO 3 - incremental review: report only new or unresolved =====
def key(f):                                        # a stable identity for a finding
    return (f["file"], f["line"], f["message"])

def incremental_review(previous, current):         # -> (labels list, resolved count)
    prev_keys = {key(p) for p in previous}
    cur_keys = {key(c) for c in current}
    labels = []
    for f in current:
        # 👉 TODO 3a: append (label, f["message"]) where label is "NEW" if key(f) not in prev_keys else "STILL OPEN"
        pass
    # 👉 TODO 3b: resolved = number of previous findings whose key is not in cur_keys
    resolved = 0
    return labels, resolved

previous = [
    {"file": "talentdesk/screening.py", "line": 1, "severity": "high", "message": "Hardcoded API key; move it to an environment variable."},
    {"file": "talentdesk/screening.py", "line": 9, "severity": "low",  "message": "Missing docstring on helper."},
]
current = envelope["structured_output"]["findings"]
labels, resolved = incremental_review(previous, current)
for label, msg in labels:
    print(f"  {label:10} - {msg[:55]}")
print("resolved since last run (not reported):", resolved)

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 3 =====
labels, resolved = incremental_review(previous, current)
label_by_msg = {msg: lab for lab, msg in labels}
assert label_by_msg["Hardcoded API key; move it to an environment variable."] == "STILL OPEN", "was in previous"
assert any(lab == "NEW" for lab, _ in labels), "the window-check finding is new this run"
assert resolved == 1, "the docstring finding was resolved and is not re-reported"
print("TODO 3 checks passed")

**TODO 4 (about 3 minutes).** Complete `gate()`, the build gate. Return `(exit_code, blocking)`
where `blocking` is the list of high-severity findings and `exit_code` is 1 if any exist, else 0. This is
the line your pipeline branches on.

In [ ]:
# ===== TODO 4 - gate the build on severity =====
def gate(findings):                                # findings -> (exit_code, blocking)
    # 👉 TODO 4a: blocking = [f for f in findings if f["severity"] == "high"]
    # 👉 TODO 4b: return (1 if blocking else 0), blocking
    return 0, []

exit_code, blocking = gate(current)
print("blocking high-severity findings:", len(blocking))
print("CI exit code:", exit_code, "(non-zero fails the build)")

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 4 =====
assert gate(current)[0] == 1, "there is a high-severity finding, so the build fails"
assert gate([{"file": "x", "line": 1, "severity": "low", "message": "m"}])[0] == 0, "no highs -> pass"
assert len(gate(current)[1]) == 1, "one blocking finding"
print("TODO 4 checks passed")

**This cell:** the live **structured-output** review (provided). It constrains the answer to the
schema via `output_config` (the API feature behind `--json-schema`) and validates what comes back.
Offline it prints the expected findings.

In [ ]:
# ===== live: schema-constrained review via the API =====
rules = "1. Every screening path checks the bar (MIN_YEARS).\n2. Never hardcode secrets.\n3. Public functions need a docstring."
code_text = 'API_KEY = "sk-ant-hardcoded"\n\ndef screen_candidate(candidate):\n    return "advance"\n'
prompt = f"Review this file against the rules and report findings.\n\nRULES:\n{rules}\n\nFILE:\n{code_text}"
if RUN_LIVE:
    try:
        resp = client.messages.create(model=MODEL, max_tokens=800,
            messages=[{"role": "user", "content": prompt}],
            extra_body={"output_config": {"format": {"type": "json_schema", "schema": REVIEW_SCHEMA}}})
        data = json.loads("".join(b.text for b in resp.content if b.type == "text"))
        jsonschema.validate(data, REVIEW_SCHEMA)
        print("schema-valid; findings:", len(data["findings"]))
        for f in data["findings"]: print("  ", f["severity"], "-", f["message"][:70])
    except Exception as e:
        print("live structured output unavailable on this account/model:", str(e)[:150])
else:
    print("[offline] expected: a schema-valid findings list (hardcoded secret, missing bar check).")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| describe behaviour in prose only | give concrete input/output examples |
| write code, then tests (or no tests) | write tests first, watch them fail, then pass |
| guess at a fuzzy spec | ask clarifying questions before coding |
| parse prose output with regex in CI | constrain to a schema; read `structured_output` |
| review in the same session that wrote the code | isolate review from generation |
| re-report every finding on each run | report only new or unresolved; suppress resolved |
| let any finding fail the build | gate on severity so only high-severity blocks |

**Lesson:** examples and tests convert a fuzzy request into a precise, checkable contract, and the
red-to-green loop keeps the code honest; when the spec is unclear, an interview beats a guess. For CI, a
schema turns an LLM answer into a dependable pipeline step, CLAUDE.md keeps the rules consistent, review
stays isolated from generation for an honest check, incremental reporting keeps re-runs quiet, and a
severity gate gives the pipeline a pass/fail signal.

---

## Recap - iterate with tests, ship through CI

| Habit / piece | In this lab | Course topic |
|---|---|---|
| Examples first | input/output pairs define correct | remove ambiguity (Lab 1) |
| Test-driven | generate tests, red, then green | code proven correct (Lab 1) |
| Interview | ask before coding a fuzzy spec | avoid wrong guesses (Lab 1) |
| Structured JSON | schema-valid `structured_output` | a contract for CI (Lab 2) |
| Review isolation | separate from generation | an honest check (Lab 2) |
| Incremental + gate | NEW vs STILL OPEN, fail on high | quiet re-runs, pass/fail signal (Lab 2) |

**Try it next:** add a partial-application example, watch the suite go red, and drive it green again. Then
add a rule to the review and confirm the incremental review labels the new finding NEW on the next run.